# Week 08 — Home exercise 1: Predict the output

**Solution proposal.**

For each snippet: what it displays, and why. Then the two broken snippets.

Every snippet here runs without an error. That is the point of the set — the mistakes this week
produces are not the kind that stop your program.

In [1]:
import pandas as pd

## a) What a `groupby` gives back

Displays `<class 'pandas.Series'>`, then a two-row Series labeled `A` and `B` holding 40 and 60, then
`40`.

Two things worth noticing. It is a **Series**, not a DataFrame — one column of answers. And `shop` is
not a column of it: the shop names are the **index**, which is why `result["A"]` works and
`result["shop"]` would raise a `KeyError`.

`reset_index()` turns it back into an ordinary two-column table when you need one.

In [2]:
sales = pd.DataFrame({
    "shop": ["A", "B", "A", "B"],
    "day":  [1, 1, 2, 2],
    "sold": [10, 20, 30, 40],
})

result = sales.groupby("shop")["sold"].sum()

print(type(result))
print(result)
print(result["A"])

<class 'pandas.Series'>
shop
A    40
B    60
Name: sold, dtype: int64
40


## b) Shape is the whole difference

Displays `(2,)` then `(4,)`.

`.mean()` **collapses**: one row per group. `.transform("mean")` **preserves shape**: one row per
original row, each holding its own group's answer.

That is why only the second can be assigned to a column of `sales`. Assigning the first would ask
pandas to line up two labels — `A` and `B` — against four row numbers, find no match anywhere, and
fill the column with `NaN`.

In [3]:
print(sales.groupby("shop")["sold"].mean().shape)
print(sales.groupby("shop")["sold"].transform("mean").shape)

sales["group_mean"] = sales.groupby("shop")["sold"].transform("mean")
sales["collapsed"] = sales.groupby("shop")["sold"].mean()
print(sales)

(2,)
(4,)
  shop  day  sold  group_mean  collapsed
0    A    1    10        20.0        NaN
1    B    1    20        30.0        NaN
2    A    2    30        20.0        NaN
3    B    2    40        30.0        NaN


## c) The rows that quietly are not there

Displays `blue 20`, `red 50` — and then `100`.

The group totals come to 70. The column comes to 100. **The 30 hours belonging to the row with no
team have been dropped**, because `groupby` leaves out every row whose key is missing.

No error, no warning, and a report that is 30% short. Passing `dropna=False` to `groupby` keeps them,
in a group of their own labeled `NaN`.

In [4]:
staff = pd.DataFrame({
    "team":  ["red", "blue", None, "red"],
    "hours": [10, 20, 30, 40],
})

print(staff.groupby("team")["hours"].sum())
print("sum of the groups:", staff.groupby("team")["hours"].sum().sum())
print("sum of the column:", staff["hours"].sum())
print()
print(staff.groupby("team", dropna=False)["hours"].sum())

team
blue    20
red     50
Name: hours, dtype: int64
sum of the groups: 70
sum of the column: 100

team
blue    20
red     50
NaN     30
Name: hours, dtype: int64


## d) Three ways to count a group

Displays group sizes of 2 and 3, then counts of 1 and 1, then means of 4.0 and 3.0.

- `.size()` counts **rows**, missing or not: `x` has two rows and `y` has three.
- `.count()` counts **values**: each group has exactly one score that is not missing.
- `.mean()` averages **the values it has**, and says nothing about how many that was. Group `y`'s
  mean of 3.0 is built from one answer out of three, and it looks exactly as solid as a mean built
  from three hundred.

The habit worth building: whenever you report a mean of a column with gaps, report its `count` beside
it.

In [5]:
survey = pd.DataFrame({
    "group": ["x", "x", "y", "y", "y"],
    "score": [4, None, 3, None, None],
})

print(survey.groupby("group").size())
print(survey.groupby("group")["score"].count())
print(survey.groupby("group")["score"].mean())

group
x    2
y    3
dtype: int64
group
x    1
y    1
Name: score, dtype: int64
group
x    4.0
y    3.0
Name: score, dtype: float64


## e) Two columns with the same name

Displays `['item', 'value_x', 'value_y']`, then a three-row table.

Both tables have a column called `value`, and it is not the key, so the result cannot have two
columns of the same name. pandas keeps both and renames them with the suffixes `_x` (from the left
table) and `_y` (from the right). Nothing is wrong yet — but in three cells' time nobody will
remember that `value_x` is a quantity and `value_y` is a price.

The fix is to say what each column is **before** the merge, with `rename`: `quantity` in one table,
`price` in the other. Then the merge produces names you would have chosen yourself.

In [6]:
prices = pd.DataFrame({"item": ["tea", "coffee"], "value": [30, 45]})
sold = pd.DataFrame({"item": ["tea", "coffee", "tea"], "value": [2, 1, 4]})

both = sold.merge(prices, on="item", how="left")

print(both.columns.tolist())
print(both)
print()

clear = sold.rename(columns={"value": "quantity"}).merge(
    prices.rename(columns={"value": "price"}), on="item", how="left"
)
print(clear)

['item', 'value_x', 'value_y']
     item  value_x  value_y
0     tea        2       30
1  coffee        1       45
2     tea        4       30

     item  quantity  price
0     tea         2     30
1  coffee         1     45
2     tea         4     30


## f) The right group, the wrong order

Displays a `change` column of `NaN, NaN, -1.0, 1.0, 3.0`.

The `groupby` is there, so no country is subtracted from another. But `.diff()` subtracts the
**previous row in the group**, and the rows are not in year order. Country A's rows arrive as 2021,
2020, 2022, so:

- A 2021 gets `NaN`, because it happens to come first,
- A 2020 gets `1.0 - 2.0 = -1.0`, a change measured backwards in time,
- A 2022 gets `4.0 - 1.0 = 3.0`, a change over two years from 2020 rather than one from 2021.

Country B is right, but only because its rows happen to arrive in order.

Grouping says *which* rows belong together. Sorting says in *what order* they come. `diff`,
`pct_change`, `shift` and `ffill` need both.

In [7]:
panel = pd.DataFrame({
    "country": ["A", "B", "A", "B", "A"],
    "year":    [2021, 2020, 2020, 2021, 2022],
    "value":   [2.0, 100.0, 1.0, 101.0, 4.0],
})

panel["change"] = panel.groupby("country")["value"].diff()
print(panel)
print()

panel = panel.sort_values(["country", "year"])
panel["change"] = panel.groupby("country")["value"].diff()
print(panel)

  country  year  value  change
0       A  2021    2.0     NaN
1       B  2020  100.0     NaN
2       A  2020    1.0    -1.0
3       B  2021  101.0     1.0
4       A  2022    4.0     3.0

  country  year  value  change
2       A  2020    1.0     NaN
0       A  2021    2.0     1.0
4       A  2022    4.0     2.0
1       B  2020  100.0     NaN
3       B  2021  101.0     1.0


## g) Dates that are still text

Displays `str`, then the rows in the order `02/11/2020`, `05/06/2022`, `10/01/2021`, then
`10/01/2021`.

The column is **text**, so both the sort and `max` compare characters from the left. `"02..."` sorts
before `"05..."` sorts before `"10..."`, and the year at the end of the string is never looked at.
So the "latest" date is the one that happens to start with the largest digits, and it is wrong
whichever way round you read the dates: 2022 is the latest year, and it is not the one returned.

Converting with the format stated makes both operations correct.

In [8]:
dates = pd.DataFrame({
    "date":  ["10/01/2021", "02/11/2020", "05/06/2022"],
    "value": [1, 2, 3],
})

print(dates["date"].dtype)
print(dates.sort_values("date"))
print(dates["date"].max())
print()

dates["date"] = pd.to_datetime(dates["date"], format="%d/%m/%Y")
print(dates.sort_values("date"))
print(dates["date"].max().date())

str
         date  value
1  02/11/2020      2
2  05/06/2022      3
0  10/01/2021      1
10/01/2021

        date  value
1 2020-11-02      2
0 2021-01-10      1
2 2022-06-05      3
2022-06-05


## Broken snippet 1: a share of the wrong total

The output is a list of shares led by `World` at 12.2%, followed by `IDA & IBRD total`, `IBRD only`
and other World Bank groupings. The five largest "countries" are not countries.

Both the numerator and the denominator are wrong, for the same reason. The file holds aggregates as
well as countries, so the denominator adds the world's emissions in many times over — once through
the countries, once through `World`, and again through every grouping. China, the largest emitter by
far, comes out at about 4%.

The fix is the lookup file: merge on `code`, drop everything whose region is `"Aggregates"`, and only
then compute the shares.

In [9]:
co2 = pd.read_csv("../data/co2_emissions.csv")

year_2023 = co2[co2["year"] == 2023].copy()
year_2023["share"] = 100 * year_2023["co2_total"] / year_2023["co2_total"].sum()

print(year_2023.sort_values("share", ascending=False)[["country", "share"]].head().round(1))
print()
print("China, broken:", year_2023[year_2023["country"] == "China"]["share"].round(1).iloc[0])

                  country  share
6167                World   12.2
2519     IDA & IBRD total    8.2
2495            IBRD only    7.9
3407  Low & middle income    7.5
3815        Middle income    7.4

China, broken: 4.1


In [10]:
info = pd.read_csv("../data/country_info.csv")

countries = co2.merge(info[["code", "region"]], on="code", how="left", validate="many_to_one")
countries = countries[countries["region"] != "Aggregates"]

year_2023 = countries[countries["year"] == 2023].copy()
year_2023["share"] = 100 * year_2023["co2_total"] / year_2023["co2_total"].sum()

print("shares add up to:", year_2023["share"].sum().round(1))
year_2023.sort_values("share", ascending=False)[["country", "share"]].head().round(1)

shares add up to: 100.0


,country,share
1127,China,34.7
5975,United States,12.3
2639,India,8.0
4751,Russian Federation,5.2
2855,Japan,2.7


Now the shares add up to 100, and China's is about 35%.

The denominator is a choice worth stating. This one is the total of the 203 countries in the file.
The World Bank's own `World` row is about four percent larger, because it covers territories the
country rows do not, so a share "of world emissions" computed against it would be slightly smaller.
Either is defensible; what is not defensible is a denominator that counts the same emissions several
times.

## Broken snippet 2: the first, not the latest

Displays one row for Norway: **2000**, with 60.2.

`drop_duplicates(subset="country")` keeps the **first** row it sees for each country and drops the
rest. The file is sorted by country and then year, so the first row is the oldest one. The code works
perfectly, and the answer is the earliest figure rather than the most recent.

Dropping rows to keep one per group only means something when you have put the row you want first.
Sort by year, newest first, and the first row per country is the latest one.

In [11]:
co2 = pd.read_csv("../data/co2_emissions.csv")

latest = co2.dropna(subset=["renew_energy"]).drop_duplicates(subset="country")
print(latest[latest["country"] == "Norway"][["country", "year", "renew_energy"]])
print()

latest = (
    co2.dropna(subset=["renew_energy"])
    .sort_values("year", ascending=False)
    .drop_duplicates(subset="country")
)
print(latest[latest["country"] == "Norway"][["country", "year", "renew_energy"]])
print()
print("latest year, across all countries:")
print(latest["year"].value_counts())

     country  year  renew_energy
4272  Norway  2000          60.2

     country  year  renew_energy
4293  Norway  2021          61.4

latest year, across all countries:
year
2021    141
2022     71
2020     43
Name: count, dtype: int64


Norway's latest figure is 61.4, from 2021. And the last cell is worth a second look: "most recent"
does not mean the same year for everyone. 141 entities' latest figure is from 2021, 71 run to 2022,
and 43 stop in 2020 — so a table of "latest figures" compares countries at different dates, and
should say so.

`drop_duplicates` also takes `keep="last"`, which does the same job on a table sorted oldest first.
Sorting the way you want and keeping the first is the version that works with every tool you know.

## The pattern behind all of these

Not one of the nine snippets raised an error. They produced:

- a Series where you expected a DataFrame (a),
- a column of `NaN` (b),
- a total that was 30% short (c),
- a mean that hides how little it is built on (d),
- column names nobody chose (e),
- changes computed in the wrong order (f),
- dates in the wrong order, and the wrong "latest" date (g),
- shares of a total that counted the world several times (broken snippet 1),
- the oldest figure presented as the newest (broken snippet 2).

The habits that catch all of them are cheap and mechanical:

- **After a groupby, check that the parts add up to the whole**, and report counts beside means.
- **Before `diff`, `ffill`, `shift` or `pct_change`, ask what the rows are a series within, and in
  what order** — then group by the first and sort by the second.
- **Before `drop_duplicates` with a `subset`, decide which row you want kept**, and sort so that it
  comes first.
- **Check the dtype of anything that looks like a date** before you sort, filter or group on it.
- **Rename before you merge** whenever both tables have a column of the same name.